# Notebook 02 — Sensitivity to Recalibration Frequency

**Research question**: How robust is the JM-XGB framework's performance to the choice of model recalibration frequency?

Shu et al. (2024) use **biannual** (semi-annual) recalibration. This notebook compares:

| Frequency | Recalibration months |
|-----------|---------------------|
| Quarterly  | Jan, Apr, Jul, Oct  |
| Semi-annual (paper) | Jan, Jul |
| Annual     | Jan                 |

We evaluate across all three portfolio models (MinVar, MV, EW) and the 0/1 individual-asset strategy.

**Hypothesis**: If the regime structure is economically meaningful rather than a statistical artefact, performance should be largely robust to moderate changes in recalibration frequency.

In [ ]:
import sys, os, pickle, warnings
from pathlib import Path

_here = Path.cwd().resolve()
_repo_root = next(
    (p for p in (_here, _here.parent, _here.parent.parent)
     if (p / "src" / "config" / "settings.py").is_file()),
    _here.parent,
)
sys.path.insert(0, str(_repo_root))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.utils.helpers import setup_logging, wealth_curve
setup_logging()

RESULTS_DIR = _repo_root / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Re-use the data already loaded in notebook 01 ────────────────────
from src.config.settings import (
    ASSETS, ASSET_TICKERS, FRED_SERIES,
    DATA_START, DATA_END, TEST_START, TEST_END
)
from src.data.loader import DataLoader
from src.data.preprocessor import DataPreprocessor

loader = DataLoader()
prices = loader.load_prices(ASSET_TICKERS, start=DATA_START, end=DATA_END)
fred   = loader.load_fred(FRED_SERIES,     start=DATA_START, end=DATA_END)

prep = DataPreprocessor()
excess_returns, rf_daily, fred_aligned = prep.prepare(prices, fred)
total_returns = prices.pct_change().reindex(excess_returns.index).ffill()

print('Data loaded. Testing period:', TEST_START, '→', TEST_END)

## 1 · Define Recalibration Schedules

In [ ]:
import os

# Full sensitivity: three calendars. For Docker/CI headless runs set THESIS_FAST_NOTEBOOKS=1
# (only the paper’s semi-annual schedule — avoids triple full JM-XGB refits).
if os.environ.get("THESIS_FAST_NOTEBOOKS", "").lower() in ("1", "true", "yes"):
    REBAL_SCHEDULES = {"Semi-Annual": (1, 7)}
else:
    REBAL_SCHEDULES = {
        "Quarterly":   (1, 4, 7, 10),
        "Semi-Annual": (1, 7),
        "Annual":      (1,),
    }

## 2 · Generate Regime Forecasts for Each Schedule

Each schedule runs the full JM-XGB framework (Algorithm 1 + 2) with a different `rebal_months` argument.  
Results are cached per schedule.

In [ ]:
from src.models.regime_framework import RegimeFramework

regime_by_schedule   = {}
lambdas_by_schedule  = {}

for sched_name, months in REBAL_SCHEDULES.items():
    cache_file = RESULTS_DIR / f'regime_{sched_name.lower().replace("-","_")}.pkl'

    if cache_file.exists():
        print(f'Loading cached [{sched_name}] …')
        with open(cache_file, 'rb') as f:
            regime_by_schedule[sched_name], lambdas_by_schedule[sched_name] = pickle.load(f)
    else:
        nb01_fc = RESULTS_DIR / 'regime_forecasts.pkl'
        nb01_lam = RESULTS_DIR / 'optimal_lambdas.pkl'
        if (
            sched_name == 'Semi-Annual'
            and months == (1, 7)
            and nb01_fc.is_file()
            and nb01_lam.is_file()
        ):
            print(f'Seeding [{sched_name}] from notebook 01 caches …')
            with open(nb01_fc, 'rb') as f:
                fc = pickle.load(f)
            with open(nb01_lam, 'rb') as f:
                lams = pickle.load(f)
            regime_by_schedule[sched_name] = fc
            lambdas_by_schedule[sched_name] = lams
            with open(cache_file, 'wb') as f:
                pickle.dump((fc, lams), f)
            print(f'  → Saved to {cache_file}')
        else:
            print(f'Running [{sched_name}] (rebal months={months}) …')
            fw = RegimeFramework(
                excess_returns = excess_returns,
                rf             = rf_daily,
                fred_aligned   = fred_aligned,
                assets         = ASSETS,
                rebal_months   = months,
            )
            fc, lams = fw.run(test_start=TEST_START, test_end=TEST_END)
            regime_by_schedule[sched_name]  = fc
            lambdas_by_schedule[sched_name] = lams

            with open(cache_file, 'wb') as f:
                pickle.dump((fc, lams), f)
            print(f'  → Saved to {cache_file}')

print('Done.')

## 3 · 0/1 Strategy — Per-Asset Sharpe by Schedule

In [ ]:
from src.models.regime_framework import _sharpe_01_strategy

er_test = excess_returns.loc[TEST_START:TEST_END]
rf_test = rf_daily.loc[TEST_START:TEST_END]

sharpe_table = pd.DataFrame(index=ASSETS, columns=list(REBAL_SCHEDULES.keys()))

for sched_name, regime_fc in regime_by_schedule.items():
    for asset in ASSETS:
        fc_a = regime_fc[asset].reindex(er_test.index).ffill()
        sr   = _sharpe_01_strategy(fc_a, er_test[asset], rf_test)
        sharpe_table.loc[asset, sched_name] = round(sr, 3)

sharpe_table = sharpe_table.astype(float)
print('=== 0/1 Strategy Sharpe Ratio by Recalibration Frequency ===')
print(sharpe_table.to_string())
sharpe_table.to_csv(RESULTS_DIR / 'sens_01_sharpe_by_schedule.csv')

In [ ]:
# Visualise
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(ASSETS))
width = 0.25
colors = ['steelblue', 'darkorange', 'green']

for i, (sched_name, color) in enumerate(zip(REBAL_SCHEDULES.keys(), colors)):
    vals = sharpe_table[sched_name].values
    ax.bar(x + i * width, vals, width, label=sched_name, color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(ASSETS, rotation=35, ha='right')
ax.axhline(0, color='k', lw=0.5)
ax.set_ylabel('Sharpe Ratio (0/1 Strategy)')
ax.set_title('Sensitivity to Recalibration Frequency — 0/1 Strategy Sharpe (2007-2023)')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sens_fig1_01sharpe_by_schedule.png', dpi=150)
plt.show()

## 4 · Portfolio Performance by Schedule

In [ ]:
from src.backtest.engine import BacktestEngine
from src.portfolio.strategies import build_strategy
from src.backtest.metrics import strategy_table

engine = BacktestEngine(
    excess_returns = excess_returns,
    total_returns  = total_returns,
    rf             = rf_daily,
    assets         = ASSETS,
)

# Strategies that use regime forecasts
regime_strategies = ['MinVar(JM-XGB)', 'EW(JM-XGB)']

port_results_by_schedule = {}

for sched_name, regime_fc in regime_by_schedule.items():
    port_cache = RESULTS_DIR / f'port_{sched_name.lower().replace("-","_")}.pkl'

    if port_cache.exists():
        with open(port_cache, 'rb') as f:
            port_results_by_schedule[sched_name] = pickle.load(f)
    else:
        strats = {name: build_strategy(name, ASSETS) for name in regime_strategies}
        results = engine.run_all(
            strategies       = strats,
            test_start       = TEST_START,
            test_end         = TEST_END,
            regime_forecasts = regime_fc,
        )
        port_results_by_schedule[sched_name] = results
        with open(port_cache, 'wb') as f:
            pickle.dump(results, f)

    metrics = {f'{name} [{sched_name}]': r['metrics']
               for name, r in port_results_by_schedule[sched_name].items()}
    print(f'\n=== {sched_name} ===')
    print(strategy_table(metrics).to_string())

## 5 · Wealth Curve Comparison

In [ ]:
for strat_name in regime_strategies:
    fig, ax = plt.subplots(figsize=(13, 5))
    colors  = ['steelblue', 'darkorange', 'green']

    for (sched_name, results), color in zip(port_results_by_schedule.items(), colors):
        if strat_name in results:
            ret = results[strat_name]['port_returns']
            wc  = wealth_curve(ret)
            ax.plot(wc.index, wc.values, label=sched_name, color=color, lw=1.5)

    ax.set_yscale('log')
    ax.set_title(f'{strat_name} — Wealth by Recalibration Frequency (2007-2023)')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_ylabel('Wealth (log scale)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.tight_layout()
    fname = f'sens_wealth_{strat_name.replace("(","").replace(")","").replace("/","").replace(" ","_")}.png'
    plt.savefig(RESULTS_DIR / fname, dpi=150)
    plt.show()

## 6 · Summary: Sharpe Ratio Heatmap

In [ ]:
import seaborn as sns

summary_rows = []
for sched_name, results in port_results_by_schedule.items():
    for strat_name, res in results.items():
        summary_rows.append({
            'Schedule': sched_name,
            'Strategy': strat_name,
            'Sharpe':   res['metrics']['Sharpe'],
            'MDD':      res['metrics']['MDD'],
            'Return':   res['metrics']['Return'],
        })

summary_df = pd.DataFrame(summary_rows)
pivot_sr   = summary_df.pivot(index='Strategy', columns='Schedule', values='Sharpe')

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    pivot_sr.astype(float),
    annot=True, fmt='.2f', cmap='RdYlGn',
    ax=ax, linewidths=0.5,
)
ax.set_title('Sharpe Ratio by Strategy and Recalibration Frequency')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sens_heatmap_sharpe.png', dpi=150)
plt.show()

print('\nCoefficient of Variation of Sharpe across schedules (lower = more robust):')
cv = pivot_sr.astype(float).std(axis=1) / pivot_sr.astype(float).mean(axis=1)
print(cv.round(3).to_string())

summary_df.to_csv(RESULTS_DIR / 'sens_summary_by_schedule.csv', index=False)

## 7 · Turnover Analysis

Higher recalibration frequency should increase model-update costs but may reduce per-period position drift.

In [ ]:
turnover_rows = []
for sched_name, results in port_results_by_schedule.items():
    for strat_name, res in results.items():
        monthly_to = res['turnover'].resample('ME').sum()
        turnover_rows.append({
            'Schedule': sched_name,
            'Strategy': strat_name,
            'Avg Monthly Turnover': monthly_to.mean(),
            'Ann Turnover':         res['metrics']['Turnover'],
        })

turnover_df = pd.DataFrame(turnover_rows)
print('=== Turnover by Schedule and Strategy ===')
print(turnover_df.pivot(index='Strategy', columns='Schedule', values='Ann Turnover').round(2).to_string())

## 8 · Formal comparison across recalibration frequencies

**Friedman test:** treats each asset as a block and tests whether 0/1 **Sharpe ratios** differ across schedules. Runs only when more than one schedule is estimated (disabled under `THESIS_FAST_NOTEBOOKS=1`).

**Pairwise schedule gap:** bootstrap distribution of mean cross-asset improvement from the **paper** semi-annual baseline vs another schedule (when available).

In [ ]:
from scipy.stats import friedmanchisquare

scheds = list(REBAL_SCHEDULES.keys())
if len(scheds) < 2:
    print("Single schedule mode (FAST): skip Friedman / pairwise schedule tests.")
else:
    cols = [sharpe_table[s].values for s in scheds]
    fr = friedmanchisquare(*cols)
    print("Friedman chi-square:", round(fr.statistic, 4), "p-value:", round(fr.pvalue, 4))
    baseline_name = "Semi-Annual" if "Semi-Annual" in scheds else scheds[0]
    for other in scheds:
        if other == baseline_name:
            continue
        delta = (sharpe_table[other] - sharpe_table[baseline_name]).astype(float)
        rng = np.random.default_rng(1)
        B = 2000
        boot_m = []
        arr = delta.values
        for _ in range(B):
            idx = rng.integers(0, len(arr), size=len(arr))
            boot_m.append(float(arr[idx].mean()))
        lo, hi = np.percentile(boot_m, [2.5, 97.5])
        print(
            f"Mean ΔSharpe ({other} − {baseline_name}): {delta.mean():.4f} "
            f"bootstrap CI [{lo:.4f}, {hi:.4f}]"
        )
    with open(RESULTS_DIR / "sens_formal_tests.txt", "w", encoding="utf-8") as f:
        f.write(f"friedman_stat={fr.statistic}\nfriedman_p={fr.pvalue}\n")